In [1]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import AutoTokenizer, EsmForMaskedLM, PretrainedConfig

from transformers import AutoModelForCausalLM
from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops
import yaml
import sys
import json

import numpy as np

/home/averma2/miniforge3/envs/exploratory_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences

In [3]:
device = "cuda"

In [4]:
with open("./esm_config35M.json", "r") as file:
    esm_config = json.load(file)
esm_config["token_dropout"] = False

esm_config = PretrainedConfig(**esm_config)

In [5]:
print(esm_config)

PretrainedConfig {
  "architectures": [
    "EsmForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.0,
  "classifier_dropout": null,
  "dtype": "float32",
  "emb_layer_norm_before": false,
  "esmfold_config": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 480,
  "initializer_range": 0.02,
  "intermediate_size": 1920,
  "is_folding_model": false,
  "layer_norm_eps": 1e-05,
  "mask_token_id": 32,
  "max_position_embeddings": 1026,
  "num_attention_heads": 20,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "position_embedding_type": "rotary",
  "token_dropout": false,
  "transformers_version": "4.57.5",
  "use_cache": true,
  "vocab_list": null,
  "vocab_size": 33
}



In [6]:
# 35 M
model_name = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM(esm_config).to(device)

<img src="./ESM2_architecture.png" width="1000"/>

In [7]:
hooked_esm_config = HookedTransformerConfig(
    n_layers=12,
    d_model=480,
    d_head=24, # 480 -> 480 = 24 * 20
    n_heads=20,
    d_mlp=1920,
    d_vocab=33,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LN",
    positional_embedding_type="rotary",
    attention_dir="bidirectional",
    post_embedding_ln=False,
    tokenizer_name=model_name
)

In [8]:
test = HookedTransformer(hooked_esm_config)

In [9]:
def rotary_embeddings(inv_freq, cfg, device=device):
    """
    https://github.com/huggingface/transformers/blob/main/src/transformers/models/esm/modeling_esm.py#L80
    """
    t = torch.arange(cfg.n_ctx).to(inv_freq.device)
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1).to(inv_freq.device)
    cos_cached = emb.cos()
    sin_cached = emb.sin()
    
    return cos_cached.to(device), sin_cached.to(device)

def get_hooked_state_dict(hf_esm_state_dict, cfg, device=device):
    """
    hugging face ESM state dict -> hooked transformer state dict
    """
    old_state_dict_keys = hf_esm_state_dict.keys()
    new_state_dict = {}

    old_to_new_weights = {
        "attention.self.query.weight":"attn.W_Q",
        "attention.self.key.weight":"attn.W_K",
        "attention.self.value.weight":"attn.W_V",
        "attention.output.dense.weight":"attn.W_O", 
    }
    old_to_new_bias = {
        "attention.self.query.bias":"attn.b_Q",
        "attention.self.key.bias":"attn.b_K",
        "attention.self.value.bias":"attn.b_V",
        "attention.output.dense.bias":"attn.b_O"
    }
    old_to_new_mlp = {
        "intermediate.dense.weight":"mlp.W_in",
        "intermediate.dense.bias":"mlp.b_in",
        "output.dense.weight":"mlp.W_out",
        "output.dense.bias":"mlp.b_out",
    }
    old_to_new_ln = {
        "attention.LayerNorm.weight":"ln1.w",
        "attention.LayerNorm.bias":"ln1.b",
        "LayerNorm.weight":"ln2.w",
        "LayerNorm.bias":"ln2.b"
    }

    # embedding matrix
    new_state_dict["embed.W_E"] = hf_esm_state_dict["esm.embeddings.word_embeddings.weight"]

    # temp unembedding matrix weight/bias (probably will delete later)
    new_state_dict["unembed.W_U"] = einops.rearrange(hf_esm_state_dict["lm_head.decoder.weight"], "d_vocab d_head -> d_head d_vocab")
    new_state_dict["unembed.b_U"] = torch.zeros(cfg.d_vocab)

    # calculating rotary embeddings (all of them are the same)
    cos_cached, sin_cached = rotary_embeddings(hf_esm_state_dict["esm.encoder.layer.0.attention.self.rotary_embeddings.inv_freq"], cfg, device)
    
    for l in range(cfg.n_layers):
        l_keys = [x for x in old_state_dict_keys if f".{l}." in x]
        old_prefix = f"esm.encoder.layer.{l}"
        new_prefix = f"blocks.{l}"

        # attn ignore = -inf
        new_state_dict[f"{new_prefix}.attn.IGNORE"] = torch.tensor(-torch.inf).to(device)
        
        # bidirectional attention, so attention should be looking everywhere
        new_state_dict[f"{new_prefix}.attn.mask"] = torch.full((cfg.n_ctx, cfg.n_ctx), True)

        # rotary embeddings
        new_state_dict[f"{new_prefix}.attn.rotary_cos"] = cos_cached
        new_state_dict[f"{new_prefix}.attn.rotary_sin"] = sin_cached
        
        # weights
        for w in old_to_new_weights.keys():
            # weights are arranged [out_features, in_features] = [n_head * d_head, d_model]
            new_weight_name = old_to_new_weights[w]
            if "output" in w:
                # [d_model d_head]
                new_state_dict[f"{new_prefix}.{new_weight_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{w}"], "d_model (n_head d_head) -> n_head d_head d_model", n_head=cfg.n_heads)
            else:
                new_state_dict[f"{new_prefix}.{new_weight_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{w}"], "(n_head d_head) d_model -> n_head d_model d_head", n_head=cfg.n_heads)
            
        #biases
        for b in old_to_new_bias.keys():
            new_bias_name = old_to_new_bias[b]
            if "output" in b:
                new_state_dict[f"{new_prefix}.{new_bias_name}"] = hf_esm_state_dict[f"{old_prefix}.{b}"]
            else:
                new_state_dict[f"{new_prefix}.{new_bias_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{b}"], "(n_head d_head) -> n_head d_head", n_head=cfg.n_heads)
            
        # mlp 
        for m in old_to_new_mlp.keys():
            # mlp are arranged [out_features, in_features] = [d_mlp, d_model]
            new_mlp_name = old_to_new_mlp[m]
            # mlp weights
            if "weight" in m:
                new_state_dict[f"{new_prefix}.{new_mlp_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{m}"], "out_feats in_feats -> in_feats out_feats")
            # mlp biases
            else:
                new_state_dict[f"{new_prefix}.{new_mlp_name}"] = hf_esm_state_dict[f"{old_prefix}.{m}"]

        # layernorms
        for ln in old_to_new_ln.keys():
            new_ln_name = old_to_new_ln[ln]
            new_state_dict[f"{new_prefix}.{new_ln_name}"] = hf_esm_state_dict[f"{old_prefix}.{ln}"]

        # Final LayerNorm
        new_state_dict["ln_final.w"] = hf_esm_state_dict["esm.encoder.emb_layer_norm_after.weight"]
        new_state_dict["ln_final.b"] = hf_esm_state_dict["esm.encoder.emb_layer_norm_after.bias"]

    return new_state_dict

In [10]:
test.load_state_dict(get_hooked_state_dict(model.state_dict(), hooked_esm_config))

<All keys matched successfully>

In [38]:
# data loading

with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
pathogens = list(config["pathogens"].keys())
pathogen_idx = 0
print(pathogens[pathogen_idx])
fasta_file = f"../../data/pathogen/{pathogens[pathogen_idx]}/alignment.fasta"
data = load_sequences(fasta_file)
sequence_names, sequences = list(zip(*list(data.items())))

influenza_h3n2_ha


In [39]:
N = 100
test_seq = sequences[N]
tokenized_seq = tokenizer(test_seq, return_tensors="pt").input_ids[:,:hooked_esm_config.n_ctx].to(device)

In [40]:
# run respective models
with torch.no_grad():
    outputs = model.forward(tokenized_seq, output_hidden_states=True)
    hf_hs = outputs.hidden_states

_, cache = test.run_with_cache(tokenized_seq)

In [41]:
for l in range(esm_config.num_hidden_layers):
    actual_output = hf_hs[l]
    hooked_output = cache[f"blocks.{l}.hook_resid_pre"]
    is_close = torch.all(torch.isclose(actual_output, hooked_output, rtol=0.1)).item()
    print(f"layer {l}: {is_close}")

layer 0: True
layer 1: True
layer 2: True
layer 3: True
layer 4: False
layer 5: True
layer 6: True
layer 7: True
layer 8: True
layer 9: True
layer 10: True
layer 11: True
